In [3]:
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holidays
 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import *

In [4]:
# data is taken from 2025 - current date 
# load the aemo data
files = sorted(glob.glob("aemo/*.csv"))
 
aemo_list = []
for f in files:
    aemo_list.append(pd.read_csv(f))
    # one csv per month, concat all together
aemo = pd.concat(aemo_list, ignore_index=True)
 
aemo["SETTLEMENTDATE"] = pd.to_datetime(aemo["SETTLEMENTDATE"])
print("no of rows:", len(aemo))
print(aemo.head())

no of rows: 169056
  REGION      SETTLEMENTDATE  TOTALDEMAND     RRP PERIODTYPE
0   VIC1 2025-01-01 00:05:00      4339.00  130.00      TRADE
1   VIC1 2025-01-01 00:10:00      4310.79  125.50      TRADE
2   VIC1 2025-01-01 00:15:00      4243.74  129.02      TRADE
3   VIC1 2025-01-01 00:20:00      4269.83  116.97      TRADE
4   VIC1 2025-01-01 00:25:00      4242.11  116.50      TRADE


In [5]:
# this is done to count the time 00:00 as the previos day 
aemo["interval_start"] = aemo["SETTLEMENTDATE"] - pd.Timedelta(minutes=5)
aemo["date"] = aemo["interval_start"].dt.normalize()
 
daily = aemo.groupby("date").agg(
    demand_mean=("TOTALDEMAND", "mean"),
    demand_max=("TOTALDEMAND", "max"),
    rrp_mean=("RRP", "mean"),
    rrp_max=("RRP", "max"),
    n_rows=("TOTALDEMAND", "size")
).reset_index()
 
print(daily.head())

        date  demand_mean  demand_max   rrp_mean  rrp_max  n_rows
0 2025-01-01  3295.657986     4488.60 -21.123993   130.00     288
1 2025-01-02  3315.613125     4721.65 -10.450764    83.98     288
2 2025-01-03  3912.291042     6153.30  46.354549   223.37     288
3 2025-01-04  4796.728611     7265.61  70.354132   160.00     288
4 2025-01-05  5328.469653     7892.55  78.097257   180.61     288


In [6]:
# load BOM data

tmax = pd.read_csv("bom/max_temp.csv")
tmin = pd.read_csv("bom/min_temp.csv")
rain = pd.read_csv("bom/rainfall.csv")
solar = pd.read_csv("bom/solar_exposure.csv")
 
 
# combine into a single date
def make_date(d):
    return pd.to_datetime(dict(year=d["Year"], month=d["Month"], day=d["Day"]))
 
tmax["date"] = make_date(tmax)
tmin["date"] = make_date(tmin)
rain["date"] = make_date(rain)
solar["date"] = make_date(solar)
 
tmax = tmax[["date", "Maximum temperature (Degree C)"]]
tmax.columns = ["date", "tmax"]
 
tmin = tmin[["date", "Minimum temperature (Degree C)"]]
tmin.columns = ["date", "tmin"]
 
rain = rain[["date", "Rainfall amount (millimetres)"]]
rain.columns = ["date", "rain"]
 
solar = solar[["date", "Daily global solar exposure (MJ/m*m)"]]
solar.columns = ["date", "solar"]
 
weather = tmax.merge(tmin, on="date").merge(rain, on="date").merge(solar, on="date")
print(weather.tail())


           date  tmax  tmin  rain  solar
4965 2026-08-06  16.8   8.1   7.2    9.2
4966 2026-08-07  16.0  10.3   0.2   10.0
4967 2026-08-08  17.7   9.8   0.0   11.1
4968 2026-08-09  14.3  11.5   1.4    3.9
4969 2026-08-10  13.1  10.1  24.4    5.3


In [7]:
# merge the two datasets together
df = daily.merge(weather, on="date", how="left")
df = df.sort_values("date").reset_index(drop=True)

# check missing values
print(df.isnull().sum())

# only tmax missing values, we can interpolate them
df["tmax"] = df["tmax"].interpolate()

date           0
demand_mean    0
demand_max     0
rrp_mean       0
rrp_max        0
n_rows         0
tmax           3
tmin           0
rain           0
solar          0
dtype: int64
